# Laboratory: Word and Lexical Tokenisation with spaCy and NLTK

## Overview and Theoretical Foundations
Tokenisation is the fundamental entry point of natural language processing (NLP) pipelines. Before a computational system can perform parsing, part-of-speech (POS) tagging, lemmatization, named entity recognition (NER), or sparse/dense vectorization, raw character streams must be partitioned into discrete linguistic units termed **tokens**.

### The Linguistic Problem of Tokenisation
While human readers instinctively recognize word boundaries, programmatic segmentation faces significant lexical ambiguities:
- **Punctuation Overloading:** Periods denote sentence boundaries, decimal numbers (`10.30`), and abbreviation markers (`U.S.`, `Mr.`, `e.g.`).
- **Contractions and Clitics:** English contractions (`We're`, `shouldn't`) combine pronouns and auxiliary verbs that must be separated for accurate grammatical role assignment.
- **Hyphenated Compounds and Web Identifiers:** Hyphens may indicate compound adjectives (`pinkish-blue`, `snail-mail`) or word breaks, while email addresses (`support@oursite.com`) and URLs must be preserved as atomic tokens.
- **Multilingual Orthographies:** Non-Latin or non-segmented scripts (such as Arabic or Chinese) present unique tokenisation challenges, where morphological clitics (prepositions and conjunctions) are affixed directly to host words.

### Toolset Architecture: spaCy vs. NLTK
- **spaCy:** Built for industrial NLP applications in Cython. Employs a deterministic, rule-based tokenizer driven by prefix, suffix, and infix regular expressions alongside language-specific exception dictionaries, returning an immutable, memory-efficient `Doc` container of `Token` views.
- **NLTK:** Designed primarily for academic research and education. Provides modular tokenizers, including the Penn Treebank standard (`word_tokenize`) and rule-based whitespace tokenizers.


---
## 1. Environment Dependencies and Tooling Setup

We utilize `spacy` for production-grade linguistic processing and `nltk` for academic reference algorithms and comparative analysis.


In [ ]:
%pip install spacy nltk


---
## 2. Ingesting spaCy Pipelines and Linguistic Models

The `spacy` library provides the core runtime engine. Language-specific capabilities require installing and loading dedicated pipeline packages.
- `en_core_web_sm` is spaCy's compact English pipeline, trained on web text (blogs, news, comments).
- Loading the model initializes a processing pipeline containing components such as `tok2vec`, `tagger`, `parser`, `attribute_ruler`, `lemmatizer`, and `ner`.


In [ ]:
import sys 

# Install spaCy small English model
!{sys.executable} -m spacy download en_core_web_sm


In [ ]:
import spacy

# Load the small English model
nlp = spacy.load("en_core_web_sm")

print("spaCy version:", spacy.__version__)
print("Active pipeline components:", nlp.pipe_names)


spaCy version: 3.8.18
Active pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


---
## 3. Fundamental Tokenisation and the `Doc` Data Model

Passing a string into the `nlp` pipeline processes the text and constructs a `Doc` object.
- The `Doc` object represents a sequence of `Token` objects.
- Crucially, spaCy preserves the exact original text and character offsets while indexing tokens, ensuring that no information is lost during segmentation.


In [ ]:
text = "Tesla is looking at buying a U.S. startup for $69 million."

# Process the text through the spaCy pipeline
doc = nlp(text)

# Inspect the returned container type
print("Container type:", type(doc))

# Iterate through discrete tokens
for token in doc:
    print(token.text)


Container type: <class 'spacy.tokens.doc.Doc'>
Tesla
is
looking
at
buying
a
U.S.
startup
for
$
69
million
.


---
## 4. Token Lexical and Morphological Attributes

Each `Token` instance exposes extensive pre-computed lexical, orthographic, and linguistic attributes:
- `token.text`: The exact Unicode character string.
- `token.shape_`: An orthographic abstraction indicating casing and digit patterns (for example, `Xxxx` for capitalized words, `d` for numbers).
- `token.is_alpha`: Boolean flag verifying whether the token consists entirely of alphabetic characters.
- `token.is_stop`: Boolean flag indicating whether the token belongs to the stopword dictionary.


In [ ]:
for token in doc:
    print(f"Token          : {token.text!r}")
    print(f"  Shape (shape_): {token.shape_}")
    print(f"  Alphabetic?   : {token.is_alpha}")
    print(f"  Stopword?     : {token.is_stop}")
    print("-" * 35)


Token          : 'Tesla'
  Shape (shape_): Xxxxx
  Alphabetic?   : True
  Stopword?     : False
-----------------------------------
Token          : 'is'
  Shape (shape_): xx
  Alphabetic?   : True
  Stopword?     : True
-----------------------------------
Token          : 'looking'
  Shape (shape_): xxxx
  Alphabetic?   : True
  Stopword?     : False
-----------------------------------
Token          : 'at'
  Shape (shape_): xx
  Alphabetic?   : True
  Stopword?     : True
-----------------------------------
Token          : 'buying'
  Shape (shape_): xxxx
  Alphabetic?   : True
  Stopword?     : False
-----------------------------------
Token          : 'a'
  Shape (shape_): x
  Alphabetic?   : True
  Stopword?     : True
-----------------------------------
Token          : 'U.S.'
  Shape (shape_): X.X.
  Alphabetic?   : False
  Stopword?     : False
-----------------------------------
Token          : 'startup'
  Shape (shape_): xxxx
  Alphabetic?   : True
  Stopword?     : False
--

---
## 5. Positional Indexing and Token Subscripting

Like standard Python sequences, a `Doc` supports integer indexing and slicing. Each index references an individual `Token` without allocating new memory.


In [ ]:
# Display the first 8 tokens with their positional indices
for i, token in enumerate(doc[:8]):
    print(f"Index {i} -> {token.text}")


Index 0 -> Tesla
Index 1 -> is
Index 2 -> looking
Index 3 -> at
Index 4 -> buying
Index 5 -> a
Index 6 -> U.S.
Index 7 -> startup


---
## 6. Subsequence Extraction via `Span` Objects

A `Span` represents a slice of contiguous tokens within a `Doc`.
- Spans are instantiated via standard Python slice notation `doc[start:end]`.
- Spans are lightweight views over the parent `Doc`; they avoid copying string buffers while providing full access to linguistic attributes.


In [ ]:
# Select tokens spanning indices 2, 3, and 4
span = doc[2:5]

print("Span text:", span.text)
print("Span type:", type(span))


Span text: looking at buying
Span type: <class 'spacy.tokens.span.Span'>


---
## 7. Complex Tokenisation: Punctuation, Contractions, and Digital Identifiers

Real-world text contains orthographic structures that defeat simplistic whitespace splitting:
- **Contractions:** `"We're"` is split into `"We"` and `"'re"`, isolating the base pronoun and the auxiliary verb.
- **Abbreviations:** `"L.A."` and `"U.S."` maintain interior punctuation without erroneous token breaks.
- **Digital Identifiers:** Email addresses (`support@oursite.com`) and URLs (`http://wwwoursite.com`) remain intact.
- **Currency Symbols:** Currency signs (`$`) are split from numeric quantities (`10.30`).


In [ ]:
examples = [
    '"We're moving to L.A."',
    '''We're here to help! Send snail-mail, email support@oursite.com
             or visit http://wwwoursite.com''',
    "A 5km NYC cab rise cost 10.30$",
    "Let's visit ST. Louis in the U.S next year"
]

for example in examples:
    print("Text  :", example)
    print("Tokens:", [token.text for token in nlp(example)])
    print("-" * 60)


Error: unterminated string literal (detected at line 2) (<string>, line 2)


---
## 8. Document Immutability and Pipeline Re-evaluation

In spaCy, `Doc` and `Token` objects are strictly immutable. You cannot directly mutate the text of a token in place.
- To modify text (for example, lexical replacement or sentiment substitution), modify the original string and pass the resulting string through the pipeline again.


In [ ]:
text1 = "My dinner was horrible."
text2 = "Your dinner was delicious."
doc1 = nlp(text1)
doc2 = nlp(text2)
print("Text 1 tokens:", [token.text for token in doc1])
print("Text 2 tokens:", [token.text for token in doc2])

# Modify the underlying string and re-execute the pipeline
revisited_text = text1.replace("horrible", "delicious")
revised_doc = nlp(revisited_text)

print("Modified text :", revisited_text)
print("New tokens    :", [token.text for token in revised_doc])


Text 1 tokens: ['My', 'dinner', 'was', 'horrible', '.']
Text 2 tokens: ['Your', 'dinner', 'was', 'delicious', '.']
Modified text : My dinner was delicious.
New tokens    : ['My', 'dinner', 'was', 'delicious', '.']


---
## 9. Tokenisation with NLTK and the Punkt Engine

NLTK approaches tokenisation through functional interfaces. The standard `word_tokenize()` function implements the Penn Treebank tokenisation specification, utilizing regular expressions to separate punctuation while keeping contractions split.


In [ ]:
import nltk

# Download required tokenisation resource packages
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import word_tokenize

example_text = """
Hello Mr. Smith, how are you doing today? The weather is great,
and Python is awesome. The sky is pinkish-blue.
You shouldn't eat cardboard.
"""

tokens_nltk = word_tokenize(example_text, language="english")
print(tokens_nltk)


[nltk_data] Error loading punkt: <urlopen error pathsec.urlopen: no
[nltk_data]     validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
[nltk_data] Error loading punkt_tab: <urlopen error pathsec.urlopen:
[nltk_data]     no validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
Error: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - '/home/youssef/.local/share/nltk_data'
    - '/home/youssef/nltk_data'
    - '/home/youssef/Projects/academic/nlp/05-spaCy-NLTK-tokenisation/.env/nltk_data'
    - '/home/youssef/Projects/academic/nlp/05-sp

---
## 10. Comparative Evaluation: `str.split()` vs. NLTK vs. spaCy

Evaluating three distinct tokenisation strategies on an ambiguous sentence:
1. **Python `str.split()`:** Naive whitespace splitting. Retains trailing punctuation (`Mr.Smith!`, `$10.30.`) and fails to isolate contractions (`shouldn't`).
2. **NLTK `word_tokenize`:** Penn Treebank conventions. Splits contractions into root and negated clitic (`should`, `n't`), but may fuse currency signs to numeric strings (`$10.30`).
3. **spaCy `nlp()`:** Industrial rule engine. Correctly isolates currency symbols (`$`, `10.30`), handles contractions (`should`, `n't`), and strips punctuation while preserving structural tokens.


In [ ]:
text = "Hello Mr.Smith! You shouldn't pay $10.30."

print("str.split():", text.split())
print("NLTK       :", word_tokenize(text, language="english"))
print("spaCy      :", [token.text for token in nlp(text)])


str.split(): ['Hello', 'Mr.Smith!', 'You', "shouldn't", 'pay', '$10.30.']
Error: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - '/home/youssef/.local/share/nltk_data'
    - '/home/youssef/nltk_data'
    - '/home/youssef/Projects/academic/nlp/05-spaCy-NLTK-tokenisation/.env/nltk_data'
    - '/home/youssef/Projects/academic/nlp/05-spaCy-NLTK-tokenisation/.env/share/nltk_data'
    - '/home/youssef/Projects/academic/nlp/05-spaCy-NLTK-tokenisation/.env/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************



---
## 11. Processing Extended Literary Text Corpora

When processing continuous prose (for example, Charles Dickens' *Hard Times*), spaCy parses token boundaries and sentence boundaries concurrently, allowing seamless traversal across paragraphs.


In [ ]:
example_text = """
Thomas Gradgrind, sir. A man of realities. A man of facts and 
calculations.
A man who proceeds upon the principle that two and two are four.
"""

doc_long = nlp(example_text)
print("Number of tokens:", len(doc_long))
print("First 20 tokens :", [token.text for token in doc_long[:20]])

# Segment into sentences
for sentence in doc_long.sents:
    print("-", sentence.text.strip())


Number of tokens: 35
First 20 tokens : ['\n', 'Thomas', 'Gradgrind', ',', 'sir', '.', 'A', 'man', 'of', 'realities', '.', 'A', 'man', 'of', 'facts', 'and', '\n', 'calculations', '.', '\n']
- Thomas Gradgrind, sir.
- A man of realities.
- A man of facts and 
calculations.
- A man who proceeds upon the principle that two and two are four.


---
## 12. Multilingual and Non-Latin Tokenisation: Arabic Script

Languages such as Arabic introduce unique morphological characteristics:
- Right-to-left (RTL) script direction.
- Highly agglutinative morphology with proclitics (prepositions such as *bi-*, conjunctions such as *wa-*) joined directly to subsequent nouns.
- Absence of capitalization.

We explore blank language initialization in spaCy (`spacy.blank("ar")`) and NLTK's `WhitespaceTokenizer` for baseline Arabic lexical segmentation.


In [ ]:
arabic_text = """
يعد الذكاء الاصطناعي من العلوم
التي يتسارع فيها مند عام 2005
"""

# Initialize a blank Arabic language pipeline in spaCy
nlp_blank_ar = spacy.blank("ar")
doc_ar = nlp_blank_ar(arabic_text)

for token in doc_ar:
    if token.text.strip():
        print(
            f"Token: {token.text!r:<12} | " 
            f"Shape: {token.shape_:<10} | " 
            f"Alphabetic: {token.is_alpha}" 
        )


Token: 'يعد'        | Shape: xxx        | Alphabetic: True
Token: 'الذكاء'     | Shape: xxxx       | Alphabetic: True
Token: 'الاصطناعي'  | Shape: xxxx       | Alphabetic: True
Token: 'من'         | Shape: xx         | Alphabetic: True
Token: 'العلوم'     | Shape: xxxx       | Alphabetic: True
Token: 'التي'       | Shape: xxxx       | Alphabetic: True
Token: 'يتسارع'     | Shape: xxxx       | Alphabetic: True
Token: 'فيها'       | Shape: xxxx       | Alphabetic: True
Token: 'مند'        | Shape: xxx        | Alphabetic: True
Token: 'عام'        | Shape: xxx        | Alphabetic: True
Token: '2005'       | Shape: dddd       | Alphabetic: False


In [ ]:
arabic_text = """
ابو عبد الله بن موسى الخوارزمي عالم رياضيات وفلك وجغرافيا مسلم
يعتبر من أوائل علماء الرياضيات المسلمين
"""

from nltk import WhitespaceTokenizer

token_ar = WhitespaceTokenizer().tokenize(arabic_text)
print("NLTK WhitespaceTokenizer (Arabic):")
print(token_ar)


NLTK WhitespaceTokenizer (Arabic):
['ابو', 'عبد', 'الله', 'بن', 'موسى', 'الخوارزمي', 'عالم', 'رياضيات', 'وفلك', 'وجغرافيا', 'مسلم', 'يعتبر', 'من', 'أوائل', 'علماء', 'الرياضيات', 'المسلمين']
